# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print("Dataset Title: ", metadata.get('name', ''))
print("Dataset Description: ", metadata.get('description', ''))

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the Croissant entities' `@id` fields to uniquely reference record sets, fields, and columns. Here is an overview:

In [ ]:
# List all available record sets and their fields by @id
record_sets = dataset.record_sets
print("Found Record Sets (by @id):")
for rs in record_sets:
    print(f"- Record Set @id: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', None)}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} | name: {getattr(field, 'name', None)} | dataType: {getattr(field, 'data_type', None)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

**Example**: Here we extract all record sets into DataFrames for easy access later.

In [ ]:
# Build a list of record set @id's
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
# Extract each record set into a DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only add if not empty
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded Record Set: {record_set_id} | Columns: {dataframes[record_set_id].columns.tolist()}")

# For demonstration, set the main record set to the first one if multiple exist
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nPreview records from main record set '{main_record_set_id}':")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations may include removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

### Example EDA Steps:
- Filter on the GAD-7 (Generalized Anxiety Disorder) total score field (by `@id`)
- Normalize the GAD-7 field
- Group by a demographic field, e.g., `level_of_education` (ensure to use its `@id`)

Before running, check available columns in the main DataFrame as their IDs may differ.

In [ ]:
# Print available columns to choose by @id
print("Main DataFrame Columns:")
print(dataframes[main_record_set_id].columns.tolist())

# Let's define the field @id for GAD-7 total score and a group_field, e.g., level_of_education
# You must replace these with the actual @id as listed above (e.g., 'gad7_total_score' -> '@id')
gad7_field_id = None
education_field_id = None
for col in dataframes[main_record_set_id].columns:
    # Heuristic: match GAD7 and education fields
    if "gad" in col.lower() and ("score" in col.lower() or "total" in col.lower()):
        gad7_field_id = col
    if "education" in col.lower():
        education_field_id = col
print(f"Chosen GAD-7 Field @id: {gad7_field_id}")
print(f"Chosen Education Field @id: {education_field_id}")

if gad7_field_id is not None:
    # Handle potential non-numeric data
    df = dataframes[main_record_set_id].copy()
    df[gad7_field_id] = pd.to_numeric(df[gad7_field_id], errors='coerce')
    threshold = 10
    filtered_df = df[df[gad7_field_id] > threshold]
    print(f"Filtered records with {gad7_field_id} > {threshold}:")
    display(filtered_df.head())
    # Normalization
    filtered_df[f"{gad7_field_id}_normalized"] = (
        filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()
    ) / filtered_df[gad7_field_id].std()
    print(f"Normalized {gad7_field_id} for filtered records:")
    display(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())
    # Grouped analysis
    if education_field_id is not None and education_field_id in df.columns:
        grouped_df = filtered_df.groupby(education_field_id).mean(numeric_only=True)
        print(f"Grouped data by {education_field_id} (education):")
        display(grouped_df)
else:
    print("No suitable numeric GAD-7 field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields.

Below is an example visualization of the GAD-7 total score distribution and its relationship with education level (if available and not missing).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if gad7_field_id is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(dataframes[main_record_set_id][gad7_field_id].dropna(), kde=True, bins=20)
    plt.title(f'Distribution of {gad7_field_id}')
    plt.xlabel('GAD-7 Total Score')
    plt.show()
    if education_field_id is not None and education_field_id in dataframes[main_record_set_id].columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=education_field_id, y=gad7_field_id, data=dataframes[main_record_set_id])
        plt.title(f'{gad7_field_id} vs {education_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable GAD-7 score field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains detailed mental health survey data, including GAD-7 anxiety scores, and demographic attributes such as education level (field IDs used: their Croissant `@id`).
- Data preview and visualizations help reveal the distribution of anxiety scores and their relation to demographic variables.
- The use of the Croissant schema via `mlcroissant` ensures reproducibility and transparency in dataset access and documentation.

Next steps may include more advanced modeling, cross-variable explorations, or exporting the curated subset for research.